# Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from cellpose import core, io, models, train

## Helper functions

In [ ]:
def to_display(image):
    """Normalize a grayscale image for display."""
    image = np.asarray(image, dtype=np.float32)
    if image.ndim == 3 and image.shape[-1] == 1:
        image = image[..., 0]
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D grayscale image, got shape {image.shape}")
    low, high = np.percentile(image, (1, 99))
    return np.clip((image - low) / (high - low + 1e-8), 0, 1)


def plot_predictions(images, predicted_masks, true_masks, image_paths):
    """Display each image beside its predicted mask and ground-truth mask."""
    n_images = len(images)
    if n_images == 0:
        raise ValueError("No images were provided for visualization")
    if not (n_images == len(predicted_masks) == len(true_masks) == len(image_paths)):
        raise ValueError("Images, predictions, labels, and paths must have equal lengths")

    fig, axes = plt.subplots(n_images, 3, figsize=(16, 4 * n_images), squeeze=False)
    titles = ("Input image", "Pretrained model prediction", "Ground truth")
    for row, (image, prediction, label, image_path) in enumerate(
        zip(images, predicted_masks, true_masks, image_paths)
    ):
        axes[row, 0].imshow(to_display(image), cmap="gray")
        axes[row, 0].set_ylabel(Path(image_path).name)
        axes[row, 1].imshow(prediction, cmap="nipy_spectral", interpolation="nearest")
        axes[row, 2].imshow(label, cmap="nipy_spectral", interpolation="nearest")
        for column, ax in enumerate(axes[row]):
            ax.set_xticks([])
            ax.set_yticks([])
            if row == 0:
                ax.set_title(titles[column])

    fig.tight_layout()
    plt.show()

## Load the prepared dataset

In [ ]:
dataset_name = "A172_splits_n500_seed42"
data_dir = Path("../data_cp")
prepared_dir = data_dir / dataset_name

train_dir = prepared_dir / "train"
validation_dir = prepared_dir / "validation"
test_dir = prepared_dir / "test"

for split_dir in (train_dir, validation_dir, test_dir):
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Prepared split directory not found: {split_dir.resolve()}")

print(f"Prepared dataset: {prepared_dir.resolve()}")

In [ ]:
output = io.load_train_test_data(train_dir = str(train_dir), 
                                 test_dir = str(test_dir), 
                                 image_filter = "",
                                 mask_filter = "_seg.npy"
                                 )
train_data, train_labels, train_names, test_data, test_labels, test_names = output

# for cellpose v3 you have to add a channel dimention:
# train_data = [np.expand_dims(a, axis=0) for a in train_data]
# train_labels = [np.expand_dims(a, axis=0) for a in train_labels]
# test_data = [np.expand_dims(a, axis=0) for a in test_data]
# test_labels = [np.expand_dims(a, axis=0) for a in test_labels]

In [ ]:
# train_data, train_labels, train_image_paths, train_mask_paths = load_split(train_dir)
# test_data, test_labels, test_image_paths, test_mask_paths = load_split(test_dir)

# These paired paths and arrays are ready for the training section below.
# train_image_mask_paths = list(zip(train_image_paths, train_mask_paths))
# test_image_mask_paths = list(zip(test_image_paths, test_mask_paths))

# print("Training image/mask paths:")
# for image_path, mask_path in train_image_mask_paths:
#     print(f"image: {image_path} | mask: {mask_path}")

## Load the pretrained Cellpose-SAM model

In [ ]:
io.logger_setup()
if not core.use_gpu():
    raise ImportError("No GPU access; use a VIBE instance with GPU support")

# Default pretrained model is cpsam_v2.
# model_backbone = "cpsam_v2"
model_backbone = "cyto"
model = models.CellposeModel(gpu=True, pretrained_model = model_backbone, )
# Other available pretrained models can be selected here:
# model = models.CellposeModel(gpu=True, pretrained_model="cpdino")
# model = models.CellposeModel(gpu=True, pretrained_model="cpdino-vitb")

## Compare pretrained predictions with test ground truth

This is a baseline from the default pretrained model, before fine-tuning it on this dataset.

In [ ]:
n_preview = min(3, len(test_data))
default_test_predictions = [model.eval(image)[0] for image in test_data[:n_preview]]

plot_predictions(
    test_data[:n_preview],
    default_test_predictions,
    test_labels[:n_preview],
    test_names[:n_preview],
)

## Train the model

Fine-tune on `train_data` and `train_labels`. The test split is passed to Cellpose for evaluation during training, then used below to compare the default and fine-tuned models against ground truth.

In [ ]:
n_epochs = 101
learning_rate = 1e-5
weight_decay = 0.1
batch_size = 64



model_name = f"{dataset_name}_{model_backbone}-finetuned_n_epochs{n_epochs}_b_size{batch_size}_w_decay{str(weight_decay)}_l_rate{str(learning_rate)}"
model_path = data_dir / ".."



new_model_path, train_losses, test_losses = train.train_seg(
                                                            model.net,
    channels=[0, 0],
                                                            train_data=train_data,
                                                            train_labels=train_labels,
                                                            test_data=test_data,
                                                            test_labels=test_labels,
                                                            batch_size=batch_size,
                                                            n_epochs=n_epochs,
                                                            learning_rate=learning_rate,
                                                            weight_decay=weight_decay,
                                                            # images sampled per epoch: all of them, at least 2
                                                            nimg_per_epoch=max(2, len(train_data)),
                                                            nimg_test_per_epoch=max(2, len(test_data)),
                                                            save_path=model_path.resolve(),
                                                            model_name=model_name,
                                                            save_each=10,
                                                            )
print("**********************************")
print(f"Fine-tuned model saved to {new_model_path}")
print("**********************************")

In [ ]:
trained_model = models.CellposeModel(gpu=True, pretrained_model=str(new_model_path))

## Plot the learning curve

Compare training and test loss by epoch to assess optimization progress and possible overfitting.

In [ ]:
train_loss_values = np.asarray(train_losses, dtype=float).reshape(-1)
test_loss_values = np.asarray(test_losses, dtype=float).reshape(-1)

if train_loss_values.size == 0:
    raise ValueError("No training losses were returned; cannot plot the learning curve")

train_epochs = np.arange(1, train_loss_values.size + 1)
test_epochs = np.arange(1, test_loss_values.size + 1)

# Keep only epochs where the test loss was actually computed
test_mask = test_loss_values > 0
test_epochs = test_epochs[test_mask]
test_loss_values = test_loss_values[test_mask]

plt.figure(figsize=(9, 5))
plt.plot(train_epochs, train_loss_values, label="Training loss")
if test_loss_values.size:
    plt.plot(
        test_epochs,
        test_loss_values,
        marker="o",
        label="Test loss",
    )
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and test loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Compare the fine-tuned model with the default and ground truth

The default predictions above were generated before fine-tuning. This visualization compares those predictions with the fine-tuned model and test ground truth.

In [ ]:
fine_tuned_test_predictions = [
    trained_model.eval(image)[0] for image in test_data[:n_preview]
]

fig, axes = plt.subplots(n_preview, 4, figsize=(20, 4 * n_preview), squeeze=False)
titles = ("Input image", "Default model", "Fine-tuned model", "Ground truth")
for row, (image, default_mask, fine_tuned_mask, true_mask, image_path) in enumerate(
    zip(
        test_data[:n_preview],
        default_test_predictions,
        fine_tuned_test_predictions,
        test_labels[:n_preview],
        test_names[:n_preview],
    )
):
    axes[row, 0].imshow(to_display(image), cmap="gray")
    axes[row, 1].imshow(default_mask, cmap="nipy_spectral", interpolation="nearest")
    axes[row, 2].imshow(fine_tuned_mask, cmap="nipy_spectral", interpolation="nearest")
    axes[row, 3].imshow(true_mask, cmap="nipy_spectral", interpolation="nearest")
    axes[row, 0].set_ylabel(Path(image_path).name)
    for column in range(4):
        axes[row, column].set_axis_off()
        if row == 0:
            axes[row, column].set_title(titles[column])

plt.tight_layout()